# Too Good To Go · login & session checks

Use this notebook to inspect your saved session, renew it, or log in again with email and PIN. **There are no reservation or cancellation calls.** Tokens and PINs are never printed by these cells.

1. Stop the dashboard server and its bots before running network cells (Ctrl+C in its terminal). Use one writer to the token file.
2. Select the project `venv` Python kernel in your IDE. If needed, run `venv/bin/python -m pip install -r requirements-notebook.txt` in a terminal, then select `venv/bin/python` as the kernel.
3. Run setup and inspection. For a new login, set `RUN_NEW_LOGIN = True` below, run the email cell once, then run the PIN cell after the email arrives.
4. Run the favorites check. Restart the dashboard and reload its page afterward.

The credential file is `temp.json` at the project root. It is written atomically with owner-only permissions. Keep this file private. Inspection reads JWT expiry without validating the signature; only an API response validates the session. A fresh login does not guarantee that an upstream 403 block is lifted.


In [ ]:
from pathlib import Path
import sys
from datetime import datetime, timezone
from getpass import getpass

PROJECT = Path.cwd()
if not (PROJECT / 'tgtg_client.py').exists():
    raise RuntimeError('Open this notebook with the project root as the working directory.')
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from tgtg_client import TgtgApiClient, token_expiry
from credential_store import load_credentials, save_credentials, credential_session

login_client = None
polling_id = None
email = None

def session_summary(credentials):
    expires = token_expiry(credentials.get('access_token'))
    return {
        'access_token_saved': bool(credentials.get('access_token')),
        'refresh_token_saved': bool(credentials.get('refresh_token')),
        'cookie_saved': bool(credentials.get('cookie')),
        'access_expiry_utc': datetime.fromtimestamp(expires, timezone.utc).isoformat() if expires else 'Not available',
        'access_expired': expires <= datetime.now(timezone.utc).timestamp() if expires else 'Unknown',
    }

print('Ready. No network request has been made.')


## 1. Inspect saved login — no network request

In [ ]:
print(session_summary(load_credentials()))


## 2. Optional: renew the saved session

Set `RUN_REFRESH = True` to make one refresh attempt and save rotated tokens. If renewal fails, use email/PIN login below. A failed HTTP request may print upstream error details; avoid sharing outputs containing private data.


In [ ]:
RUN_REFRESH = False
if RUN_REFRESH:
    with credential_session() as client:
        client.refresh_credentials()
    print('Session renewed and saved.')
    print(session_summary(load_credentials()))
else:
    print('Refresh skipped. Continue to email/PIN login if needed.')


## 3. New login: request an email PIN

Set the flag to `True` and run once. Enter your email in the hidden input prompt. The same Python client is retained for PIN verification, preserving its cookies. Requesting a new PIN replaces the previous pending login.


In [ ]:
RUN_NEW_LOGIN = False
if RUN_NEW_LOGIN:
    email = getpass('Too Good To Go email: ').strip()
    if not email or '@' not in email:
        raise ValueError('Enter a valid email address.')
    login_client = TgtgApiClient()
    auth_response = login_client.request_auth_code(email)
    polling_id = auth_response.get('polling_id')
    if not polling_id:
        login_client = None
        raise RuntimeError('No polling ID returned. Stop here and try login in the official app.')
    print('PIN requested. Check your email, then run the next cell.')
else:
    print('New login skipped. Set RUN_NEW_LOGIN = True to request a PIN.')


## 4. Verify the PIN and save the new session

In [ ]:
if login_client is not None and polling_id:
    pin = getpass('Email PIN: ').strip()
    try:
        new_credentials = login_client.submit_pin(email, pin, polling_id)
        save_credentials(new_credentials)
        print('Login saved to temp.json. Raw tokens are not displayed.')
        print(session_summary(new_credentials))
        polling_id = None
        login_client = None
        del new_credentials
    finally:
        pin = None
else:
    print('No pending login. Run the email cell first, or skip if you refreshed successfully.')


## 5. Read-only favorites check

Set `RUN_FAVORITES = True`. This makes a fresh request rather than displaying the dashboard's cached list. Automatic token rotation is saved even if a later request fails. Only a count is printed; it does not reserve anything.


In [ ]:
RUN_FAVORITES = False
if RUN_FAVORITES:
    with credential_session() as client:
        favorites = client.get_favorites()
    print(f'Live favorites request succeeded: {len(favorites)} bags returned.')
    print(session_summary(load_credentials()))
else:
    print('Favorites request skipped.')


## Return to the dashboard

Start `uvicorn app:app --reload` from the project root and reload the browser. Use **Refresh** in Favorites to make a fresh request. **Reconnect account** in Authentication is also available if you prefer the web flow.

- **401:** authentication failed. An old access token cannot be reused indefinitely; its refresh token may also be invalid. Use a new email/PIN login.
- **403:** upstream blocked the request. Stop repeated attempts and check the official app. This is different from token expiry.
- Cached favorites are not evidence that a reservation request can authenticate now.
